# 🎙️ JARVIS ovozi — RVC model o'qitish (Google Colab)

Bu notebook sizning **Jarvis ovoz namunalaringiz**dan sun'iy ovoz modeli yaratadi. Model Google Colab'ning **bepul T4 GPU**sida o'qitiladi va natijada `jarvis.pth` hamda `jarvis.index` fayllari **Google Drive**ga saqlanadi.

Texnologiya: [Applio](https://github.com/IAHispano/Applio) (RVC asosidagi ochiq manbali vosita).

---

## 📋 Qadamlar jadvali

| # | Qadam | Qancha vaqt |
|---|---|---|
| 1 | Ovoz namunalari qayta ishlanadi | 2-5 daqiqa |
| 2 | Xususiyatlar ajratiladi | 10-30 daqiqa |
| 3 | Index fayl yaratiladi | 1-3 daqiqa |
| 4 | Model o'qitiladi (200 epox) | 1-3 soat |
| 5 | Model Drive'ga saqlanadi | 1 daqiqa |

**Diqqat:** o'qitish paytida brauzer yorlig'ini (tab) yopmang, aks holda Colab uzilib qolishi mumkin.

## ⚠️ Boshlashdan oldin (muhim!)

1. **GPU'ni yoqing:** yuqori menyuda `Runtime` → `Change runtime type` → `T4 GPU` ni tanlang.
2. **Ovoz namunalarini tayyorlang:** Jarvis ovozidagi **10-30 daqiqa** toza nutqni (musiqa va shovqinsiz) Google Drive'ning `MyDrive/Jarvis_Dataset` papkasiga yuklang (keyingi hujayralarda batafsil ko'rsatma bor).
3. **Hujayralarni tartib bilan** ishga tushiring — yuqoridan pastga, har bir hujayraning chap tomonidagi ▶ tugmasini bosib.

In [ ]:
# GPU mavjudligini tekshirish
import torch

if torch.cuda.is_available():
    print("✅ GPU tayyor:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU yo'q! Menyu: Runtime → Change runtime type → T4 GPU tanlang,")
    print("   so'ng bu hujayrani qayta ishga tushiring.")

In [ ]:
# Google Drive'ni ulash
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Ovoz namunalari papkasi (bu yerga .wav fayllarni qo'yasiz)
DATASET_DIR = Path('/content/drive/MyDrive/Jarvis_Dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Tayyor model saqlanadigan papka
EXPORT_DIR = Path('/content/drive/MyDrive/ApplioExported')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Drive ulandi!')
print('📂 Ovoz namunalari:', DATASET_DIR)
print('📤 Model saqlanadi:', EXPORT_DIR)

## 🎵 Ovoz namunalarini tayyorlash

Model sifati **namunalar sifatiga** bevosita bog'liq. Mana talablar:

- **Umumiy davomiyligi:** kamida 10 daqiqa, ideal holda 20-30 daqiqa.
- **Toza nutq:** faqat Jarvis ovozi, fon musiqasi va shovqinsiz. Filmdan olingan audio bo'lsa, avval ovozni ajratib oling (masalan, [vocalremover.org](https://vocalremover.org) yoki UVR5 kabi bepul vositalar bilan musiqa va fon shovqinini olib tashlang).
- **Format:** `.wav` fayllar (agar .mp3 bo'lsa, avval .wav ga o'tkazing — masalan [online-audio-converter.com](https://online-audio-converter.com)).
- **Joyi:** barcha fayllarni Google Drive'dagi `MyDrive/Jarvis_Dataset` papkasiga yuklang (fayl nomlari muhim emas).

Namunalar tayyor bo'lgach, pastdagi hujayralarni davom ettiring.

In [ ]:
# @title ⚙️ Applio'ni o'rnatish (5-10 daqiqa)
# @markdown Faqat birinchi marta ishga tushiriladi. Runtime qayta ishga tushsa, bu hujayrani qayta ishga tushiring.
from multiprocessing import cpu_count

cpu_cores = cpu_count()
print('CPU yadrolari:', cpu_cores)

%cd /content
!git config --global advice.detachedHead false
!git clone https://github.com/IAHispano/Applio

%cd /content/Applio
!apt update -y -qq
!apt install -y -qq portaudio19-dev > /dev/null 2>&1

print("📦 Kutubxonalar o'rnatilmoqda (bir necha daqiqa)...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!~/.local/bin/uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match

print('🤖 Asosiy modellar yuklab olinmoqda (hubert, rmvpe, HiFi-GAN)...')
!python core.py prerequisites --models --pretraineds-hifigan --exe
!cp assets/config_template.json assets/config.json

print("✅ Applio tayyor! Keyingi hujayraga o'ting.")

In [ ]:
# @title 🎛️ Model parametrlari
# @markdown Model nomini o'zgartirmang — keyingi qadamlar shu nomni ishlatadi.
model_name = "jarvis"  # @param {type:"string"}
sample_rate = "40k"  # @param ["32k", "40k", "48k"] {allow-input: false}
sr = int(sample_rate.rstrip("k")) * 1000
print('Model nomi:', model_name)
print('Namuna chastotasi:', sr, 'Hz')

# Ovoz namunalari papkasini tekshirish
from pathlib import Path

dataset_path = "/content/drive/MyDrive/Jarvis_Dataset"  # @param {type:"string"}
wav_files = sorted(Path(dataset_path).glob("*.wav")) + sorted(Path(dataset_path).glob("*.flac"))
print('🎵 Topilgan fayllar soni:', len(wav_files))
if not wav_files:
    print("⚠️ Papkada ovoz fayli topilmadi! Yuqoridagi 'Ovoz namunalarini tayyorlash' bo'limiga qarang.")
elif len(wav_files) < 10:
    print('⚠️ Kamida 10 ta fayl tavsiya etiladi (jami 10-30 daqiqa nutq).')

In [ ]:
# @title 1-qadam: Ovoz namunalarini qayta ishlash (2-5 daqiqa)
# @markdown Ovozlar 3 soniyalik bo'laklarga bo'linadi va tayyorlanadi.
cut_preprocess = "Automatic"  # @param ["Skip", "Simple", "Automatic"] {allow-input: false}
chunk_len = 3  # @param {type:"slider", min:0.5, max:5.0, step:0.5}
overlap_len = 0.3  # @param {type:"slider", min:0, max:0.5, step:0.1}
noise_reduction = False  # @param{type:"boolean"}

%cd /content/Applio
import subprocess

def run_stream(cmd):
    """Buyruqni jonli chiqish bilan ishga tushiradi."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    return proc.returncode

pp_cmd = [
    "python", "core.py", "preprocess",
    "--model-name", model_name,
    "--dataset-path", dataset_path,
    "--sample-rate", str(sr),
    "--cpu-cores", str(cpu_cores),
    "--cut-preprocess", cut_preprocess,
    "--chunk-len", str(chunk_len),
    "--overlap-len", str(overlap_len),
    "--normalization-mode", "none",
]
if noise_reduction:
    pp_cmd.append("--noise-reduction")

code = run_stream(pp_cmd)
print("✅ 1-qadam tugadi!" if code == 0 else "❌ Xatolik yuz berdi (yuqoridagi logga qarang)")

In [ ]:
# @title 2-qadam: Xususiyatlarni ajratib olish (10-30 daqiqa)
f0_method = "rmvpe"  # @param ["crepe", "crepe-tiny", "rmvpe"] {allow-input: false}
embedder_model = "contentvec"  # @param ["contentvec", "spin-v2"] {allow-input: false}

%cd /content/Applio
code = run_stream([
    "python", "core.py", "extract",
    "--model-name", model_name,
    "--f0-method", f0_method,
    "--sample-rate", str(sr),
    "--cpu-cores", str(cpu_cores),
    "--gpu", "0",
    "--embedder-model", embedder_model,
    "--include-mutes", "2",
])
print("✅ 2-qadam tugadi!" if code == 0 else "❌ Xatolik yuz berdi")

In [ ]:
# @title 3-qadam: Index fayl yaratish (1-3 daqiqa)
%cd /content/Applio
code = run_stream([
    "python", "core.py", "index",
    "--model-name", model_name,
    "--index-algorithm", "Auto",
])
print("✅ 3-qadam tugadi!" if code == 0 else "❌ Xatolik yuz berdi")

In [ ]:
# @title 4-qadam: Modelni o'qitish (1-3 soat) 🚀
# @markdown Eng uzun qadam. **Brauzer yorlig'ini ochiq tuting!**
# @markdown Xotira yetmasa ("CUDA out of memory") batch_size ni 4 ga kamaytiring.
total_epoch = 200  # @param {type:"integer"}
batch_size = 8  # @param {type:"slider", min:1, max:25, step:0}
save_every_epoch = 10  # @param {type:"slider", min:1, max:100, step:0}
save_only_latest = True  # @param{type:"boolean"}
cache_data_in_gpu = False  # @param{type:"boolean"}
vocoder = "HiFi-GAN"  # @param ["HiFi-GAN", "RefineGAN"] {allow-input: false}

%cd /content/Applio
train_cmd = [
    "python", "core.py", "train",
    "--model-name", model_name,
    "--save-every-epoch", str(save_every_epoch),
    "--total-epoch", str(total_epoch),
    "--sample-rate", str(sr),
    "--batch-size", str(batch_size),
    "--gpu", "0",
    "--vocoder", vocoder,
    "--pretrained",
]
if save_only_latest:
    train_cmd.append("--save-only-latest")
if cache_data_in_gpu:
    train_cmd.append("--cache-data-in-gpu")

code = run_stream(train_cmd)
print("✅ O'qitish tugadi!" if code == 0 else "❌ Xatolik yuz berdi")

In [ ]:
# @title 5-qadam: Modelni Google Drive'ga saqlash (1 daqiqa)
# @markdown `jarvis.pth` va `jarvis.index` fayllari Drive'ga ko'chiriladi.
from pathlib import Path
import shutil

logs_folder = Path(f"/content/Applio/logs/{model_name}")
if not logs_folder.is_dir():
    raise FileNotFoundError(f"Model papkasi topilmadi: {logs_folder}")

# Eng so'nggi o'qitilgan modelni topamiz
pth_files = list(logs_folder.glob(f"{model_name}_*e_*s.pth"))
if not pth_files:
    raise FileNotFoundError("O'qitilgan model fayli topilmadi — avval 4-qadamni tugating!")
pth_src = max(pth_files, key=lambda p: p.stat().st_mtime)  # eng so'nggi (mtime bo'yicha)
print("📄 Eng so'nggi model:", pth_src.name)

index_src = logs_folder / f"{model_name}.index"
if not index_src.exists():
    print("⚠️ .index fayli topilmadi — 3-qadamni qayta ishga tushiring!")

# Drive'ga ko'chirish (jarvis.pth deb qayta nomlanadi)
dest = EXPORT_DIR / model_name
dest.mkdir(parents=True, exist_ok=True)
shutil.copy2(pth_src, dest / "jarvis.pth")
if index_src.exists():
    shutil.copy2(index_src, dest / "jarvis.index")

# Zaxira nusxasi (zip) — o'qitishni davom ettirish kerak bo'lsa
backup_zip = f"/content/{model_name}_backup.zip"
shutil.make_archive(f"/content/{model_name}_backup", "zip", logs_folder.parent, model_name)
shutil.copy2(backup_zip, EXPORT_DIR / f"{model_name}_backup.zip")

print("\n✅ Model Drive'ga saqlandi:")
for f in sorted(dest.iterdir()):
    print(f"   📁 {EXPORT_DIR}/{model_name}/{f.name}  ({f.stat().st_size / 1024 / 1024:.1f} MB)")
print("\nEndi bu 2 ta faylni (jarvis.pth va jarvis.index) kompyuteringizga yuklab oling.")
print("Keyingi qadamlar: QOLLANMA.md faylida yozilgan.")

## 🏁 Tayyor!

Google Drive → `MyDrive/ApplioExported/jarvis/` papkasida quyidagilar bor:

- **`jarvis.pth`** — model og'irliklari (asosiy fayl)
- **`jarvis.index`** — xususiyatlar indeksi (ovoz sifatini yaxshilaydi)
- **`jarvis_backup.zip`** — zaxira nusxa (kerak bo'lmasa o'chirib tashlang)

Bu 2 faylni kompyuteringizga yuklab oling va loyihaning `data/models/rvc/jarvis/` papkasiga joylashtiring. Keyingi bosqichlar loyihadagi **`rvc_training/QOLLANMA.md`** faylida batafsil yozilgan.